In [2]:
import traci
import os
import sys
import random
import torch
import torch.nn as nn
import torch.optim as optim

# Close previous connection
if traci.isLoaded():
    traci.close()

# SUMO setup
if 'SUMO_HOME' in os.environ:
    tools = os.path.join(os.environ['SUMO_HOME'], 'tools')
    sys.path.append(tools)
else:
    raise Exception("SUMO_HOME not set")

sumoBinary = "sumo-gui"
sumoCmd = [sumoBinary, "-c", "../sumo_files/simple.sumocfg"]

traci.start(sumoCmd)

tls = traci.trafficlight.getIDList()[0]
print("Using TLS:", tls)

# ---------------- STATE ----------------
def get_state(tls_id):
    lanes = traci.trafficlight.getControlledLanes(tls_id)
    total = 0
    for lane in lanes:
        total += traci.lane.getLastStepVehicleNumber(lane)
    return float(total)

# ---------------- REWARD ----------------
def get_reward(state):
    return -state

# ---------------- DQN MODEL ----------------
class DQN(nn.Module):
    def __init__(self):
        super(DQN, self).__init__()
        self.fc1 = nn.Linear(1, 32)
        self.fc2 = nn.Linear(32, 32)
        self.fc3 = nn.Linear(32, 2)

    def forward(self, x):
        x = torch.relu(self.fc1(x))
        x = torch.relu(self.fc2(x))
        return self.fc3(x)

model = DQN()
optimizer = optim.Adam(model.parameters(), lr=0.001)
loss_fn = nn.MSELoss()

# ---------------- PARAMETERS ----------------
gamma = 0.9
epsilon = 0.2

# ---------------- ACTION ----------------
def choose_action(state):
    if random.random() < epsilon:
        return random.choice([0, 1])

    state_tensor = torch.FloatTensor([[state]])
    q_values = model(state_tensor)

    return torch.argmax(q_values).item()

# ---------------- SIMULATION ----------------
current_phase = 0

print("Running DQN...")

for step in range(300):
    traci.simulationStep()

    state = get_state(tls)
    action = choose_action(state)

    # Apply action
    if action == 1:
        current_phase = 2 if current_phase == 0 else 0
        traci.trafficlight.setPhase(tls, current_phase)

    reward = get_reward(state)

    next_state = get_state(tls)

    # Convert to tensors
    state_tensor = torch.FloatTensor([[state]])
    next_state_tensor = torch.FloatTensor([[next_state]])

    # Get Q values
    q_values = model(state_tensor)
    next_q_values = model(next_state_tensor)

    target = q_values.clone().detach()
    target[0][action] = reward + gamma * torch.max(next_q_values).item()

    # Train
    loss = loss_fn(q_values, target)

    optimizer.zero_grad()
    loss.backward()
    optimizer.step()

    print(f"Step: {step} | State: {state} | Action: {action} | Reward: {reward}")

traci.close()

Using TLS: A1
Running DQN...
Step: 0 | State: 0.0 | Action: 1 | Reward: -0.0
Step: 1 | State: 0.0 | Action: 1 | Reward: -0.0
Step: 2 | State: 0.0 | Action: 1 | Reward: -0.0
Step: 3 | State: 0.0 | Action: 1 | Reward: -0.0
Step: 4 | State: 3.0 | Action: 0 | Reward: -3.0
Step: 5 | State: 6.0 | Action: 0 | Reward: -6.0
Step: 6 | State: 9.0 | Action: 0 | Reward: -9.0
Step: 7 | State: 9.0 | Action: 0 | Reward: -9.0
Step: 8 | State: 12.0 | Action: 0 | Reward: -12.0
Step: 9 | State: 12.0 | Action: 0 | Reward: -12.0
Step: 10 | State: 12.0 | Action: 0 | Reward: -12.0
Step: 11 | State: 12.0 | Action: 0 | Reward: -12.0
Step: 12 | State: 12.0 | Action: 0 | Reward: -12.0
Step: 13 | State: 12.0 | Action: 1 | Reward: -12.0
Step: 14 | State: 12.0 | Action: 1 | Reward: -12.0
Step: 15 | State: 12.0 | Action: 1 | Reward: -12.0
Step: 16 | State: 6.0 | Action: 1 | Reward: -6.0
Step: 17 | State: 3.0 | Action: 0 | Reward: -3.0
Step: 18 | State: 3.0 | Action: 1 | Reward: -3.0
Step: 19 | State: 3.0 | Action: 1 